<a href="https://colab.research.google.com/github/DeveshValluru/PytorchCodingModelsFromScratch/blob/main/TorchBasics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import torch

w = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(2.0)

y = w * x
loss = (y - 5) ** 2

# Walk the chain
node = loss.grad_fn
while node is not None:
    print(node)
    parents = node.next_functions
    if parents:
        node = parents[0][0]  # follow first parent
    else:
        node = None

In [34]:
class LeakyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return torch.where(input > 0, input, 0.01 * input)

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        local_derivative = torch.where(input > 0,
                                        torch.ones_like(input),
                                        0.01 * torch.ones_like(input))
        return grad_output * local_derivative

In [35]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = LeakyReLU.apply(x)         # [0, 0, 0, 1, 2]
y.sum().backward()
print(x.grad)                # [0, 0, 0, 1, 1]

tensor([0.0100, 0.0100, 0.0100, 1.0000, 1.0000])


In [36]:
class Sigmoid(torch.autograd.Function):
  @staticmethod
  def forward(ctx, input):
    output = 1/(1+torch.exp(-input))
    ctx.save_for_backward(output)
    return output

  @staticmethod
  def backward(ctx,grad_output):
    output, = ctx.saved_tensors
    local_derivative = output * (1-output)
    return grad_output * local_derivative

In [37]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = Sigmoid.apply(x)         # [0, 0, 0, 1, 2]
y.sum().backward()
print(x.grad)                # [0, 0, 0, 1, 1]

tensor([0.1050, 0.1966, 0.2500, 0.1966, 0.1050])


In [38]:
class Tanh(torch.autograd.Function):
  @staticmethod
  def forward(ctx, input):
    output = torch.tanh(input)
    ctx.save_for_backward(output)
    return output

  @staticmethod
  def backward(ctx, grad_output):
    output, = ctx.saved_tensors
    local_derivative = 1- output**2
    return grad_output * local_derivative

In [39]:
x = torch.tensor([-1.0, 0.0, 0.5, 2.0], requires_grad=True)
y = Tanh.apply(x)

y.sum().backward()
print(x.grad)

tensor([0.4200, 1.0000, 0.7864, 0.0707])


In [40]:
print(torch.cuda.is_available())

True


In [41]:
print(torch.cuda.device_count())

1


In [42]:
print(torch.cuda.get_device_name(0))

Tesla T4


In [43]:
x= torch.tensor([1.0,2.0])

In [44]:
print(x.device)

cpu


In [45]:
x_gpu = torch.tensor([1.0,2.0], device= 'cuda')

In [46]:
print(x_gpu.device)

cuda:0


In [47]:
x = torch.randn(1000,1000)

In [48]:
print(x)

tensor([[ 0.8450,  0.0974,  1.4154,  ..., -2.8106,  1.3566, -0.1488],
        [-0.2043,  0.3526, -2.2600,  ..., -0.4636,  0.5004, -1.0827],
        [ 0.5363,  0.6386,  1.0029,  ...,  1.6828, -1.3072,  0.1591],
        ...,
        [ 1.5087,  1.0069,  0.7864,  ...,  0.4610,  1.4263,  0.8752],
        [ 1.1024, -1.7888,  0.4605,  ...,  0.3002, -0.5514, -0.1555],
        [-0.9466,  0.1192,  0.4043,  ...,  0.5940, -1.2787, -0.5861]])


In [49]:
x_gpu = x.to('cuda')

In [50]:
print(x_gpu.device)

cuda:0


In [51]:
x_cpu = x_gpu.to('cpu')

In [52]:
print(x_cpu.device)

cpu


In [53]:
import torch.nn as nn

In [54]:
model = nn.Linear(100,10)
print(next(model.parameters()).device)

cpu


In [55]:
model = model.cuda()

In [56]:
print(next(model.parameters()).device)

cuda:0


In [57]:
x = torch.randn(32, 100).to('cuda')
output = model(x)                        # works — both on GPU
print(output.device)                     # cuda:0

cuda:0


In [58]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')